In [1]:
!pip install torch transformers peft accelerate bitsandbytes langchain langchain-community langchain-chroma chromadb sentence-transformers numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 51.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 103.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 85.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
"""
Copy a Chroma DB (or any folder) out of the read-only /kaggle/input mount
into the writable /kaggle/working directory.

Run this ONCE per session before loading the DB with Chroma(persist_directory=...).
"""

import os
import shutil

# EDIT THESE TWO PATHS to match your setup
SOURCE_DIR = "/kaggle/input/datasets/teslaincarnate/chroma/scd_guidelines"
DEST_DIR = "/kaggle/working/vectordb/scd_guidelines"

def copy_db():
    if not os.path.exists(SOURCE_DIR):
        print(f"ERROR: Source not found: {SOURCE_DIR}")
        return

    if os.path.exists(DEST_DIR):
        size = sum(f.stat().st_size for f in os.scandir(DEST_DIR) if f.is_file())
        print(f"Destination already exists ({DEST_DIR}), skipping copy.")
        print("Delete it first with: shutil.rmtree(DEST_DIR) if you want a fresh copy.")
        return

    src_size_mb = sum(
        os.path.getsize(os.path.join(dirpath, f))
        for dirpath, _, filenames in os.walk(SOURCE_DIR)
        for f in filenames
    ) / (1024 * 1024)
    print(f"Copying {src_size_mb:.1f} MB from:\n  {SOURCE_DIR}\nto:\n  {DEST_DIR}")

    os.makedirs(os.path.dirname(DEST_DIR), exist_ok=True)
    shutil.copytree(SOURCE_DIR, DEST_DIR)

    print("Done. You can now load it with:")
    print(f'  Chroma(persist_directory="{DEST_DIR}", embedding_function=...)')

if __name__ == "__main__":
    copy_db()

Copying 33.7 MB from:
  /kaggle/input/datasets/teslaincarnate/chroma/scd_guidelines
to:
  /kaggle/working/vectordb/scd_guidelines
Done. You can now load it with:
  Chroma(persist_directory="/kaggle/working/vectordb/scd_guidelines", embedding_function=...)


In [3]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_chroma import Chroma
import numpy as np
from sentence_transformers import CrossEncoder

            # ── config ──────────────────────
BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
ADAPTER = "TeslaInch/scd-phi35-adapter-v8"  # best adapter
VECTORDB_PATH = "/kaggle/working/vectordb/scd_guidelines"
if not os.path.exists(VECTORDB_PATH):
    VECTORDB_PATH = "data/vectordb/scd_guidelines"

# output path also needs a local fallback — /kaggle/working only exists on Kaggle
OUTPUT_DIR = "/kaggle/working"
if not os.path.exists(OUTPUT_DIR):
    OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "rag_test_results.json")

SYSTEM = (
    "You are a medical AI assistant specialised in sickle cell disease. "
    "Answer clinical questions accurately using the provided guidelines. "
    "Always mention which guideline informed your answer."
)


/tmp/ipykernel_58/857292745.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceBgeEmbeddings


In [ ]:
# ── load model ────────────────────────────────────────────────────────────────
def load_model():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.bfloat16,
        attn_implementation="sdpa",  # switch to "sdpa" for faster inference if you don't need attention outputs
    )
    model = PeftModel.from_pretrained(model, ADAPTER)
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.pad_token = tokenizer.eos_token
    # left-padding is required for decoder-only generation. Right-padding with
    # pad_token == eos_token can make the model treat the sequence as already
    # finished, degrading or truncating generation.
    tokenizer.padding_side = "left"

    return model, tokenizer

# ── load vector database ──────────────────────────────────────────────────────
def load_vectordb():
    embeddings = HuggingFaceBgeEmbeddings(
        model_name="BAAI/bge-large-en-v1.5",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
        query_instruction="Represent this sentence for searching relevant passages: "
    )
    vectordb = Chroma(
        persist_directory=VECTORDB_PATH,
        embedding_function=embeddings
    )
    # Return the embeddings object we built ourselves, rather than having
    # callers reach into vectordb._embedding_function later. That attribute
    # is a private implementation detail of langchain_chroma.Chroma and isn't
    # part of its public API — it could be renamed or restructured in a
    # future release without a deprecation warning. Holding our own
    # reference is stable regardless of what Chroma does internally.
    return vectordb, embeddings

# ── diversity-aware selection (MMR) ─────────────────────────────────────────
def mmr_select(candidate_vectors, relevance_scores, sources, k=3,
                lambda_param=0.7, same_source_penalty=0.15):
    """
    Greedy Maximal Marginal Relevance selection.

    At each step, pick the candidate that maximizes:
        lambda * relevance(i) - (1 - lambda) * max_sim(i, already_selected)

    This penalizes chunks that are semantically close to ones already picked,
    even if they come from *different* source files (e.g. two guidelines
    both quoting the same WHO recommendation) — which plain filename dedup
    can't catch. `same_source_penalty` adds a small extra nudge away from
    repeating a filename when relevance/similarity are close, since exact
    same-document repeats are still usually less useful for citation variety.

    candidate_vectors: (n, d) array of L2-normalized embedding vectors, so
        dot product == cosine similarity.
    relevance_scores: (n,) array, higher = more relevant (e.g. cross-encoder
        scores, min-max normalized to [0, 1] beforehand).
    sources: list of length n, source filename per candidate.
    """
    n = len(relevance_scores)
    k = min(k, n)
    selected = []
    remaining = list(range(n))

    while len(selected) < k:
        best_idx, best_score = None, -np.inf
        for i in remaining:
            if selected:
                sim_to_selected = max(
                    float(np.dot(candidate_vectors[i], candidate_vectors[j]))
                    for j in selected
                )
            else:
                sim_to_selected = 0.0

            penalty = same_source_penalty if (
                selected and sources[i] in {sources[j] for j in selected}
            ) else 0.0

            mmr_score = (
                lambda_param * relevance_scores[i]
                - (1 - lambda_param) * sim_to_selected
                - penalty
            )
            if mmr_score > best_score:
                best_score, best_idx = mmr_score, i

        selected.append(best_idx)
        remaining.remove(best_idx)

    return selected

# ── RAG answer function ───────────────────────────────────────────────────────
def answer_with_rag(pipe, vectordb, embeddings, cross_encoder, question, case=""):
    # retrieve relevant context — widened candidate pool (was k=10) so MMR
    # has enough distinct material to actually select diverse chunks from
    search_query = f"{case} {question}" if case else question
    retrieved = vectordb.similarity_search(search_query, k=25)

    if not retrieved:
        # No guideline retrieved at all — don't let the model answer as if it
        # had grounding. Surface this explicitly instead of silently
        # generating from an empty context block.
        return (
            "No relevant guideline could be retrieved for this question. "
            "Answer withheld to avoid ungrounded clinical output.",
            []
        )

    # rerank with cross-encoder for better relevance ordering than raw
    # embedding similarity
    pairs = [[search_query, doc.page_content] for doc in retrieved]
    raw_scores = np.array(cross_encoder.predict(pairs), dtype=float)

    # min-max normalize relevance to [0, 1] so it's on the same scale as
    # cosine similarity for the MMR trade-off
    score_range = raw_scores.max() - raw_scores.min()
    relevance = (
        (raw_scores - raw_scores.min()) / score_range
        if score_range > 0 else np.ones_like(raw_scores)
    )

    # embed candidates for similarity comparisons using the embeddings
    # object we constructed in load_vectordb() and passed in explicitly —
    # a stable reference we control, not a private Chroma attribute.
    # encode_kwargs sets normalize_embeddings=True on this embedding
    # function, so a plain dot product below gives cosine similarity directly.
    candidate_vectors = np.array(
        embeddings.embed_documents([doc.page_content for doc in retrieved])
    )
    sources_list = [doc.metadata.get("source", "unknown") for doc in retrieved]

    selected_indices = mmr_select(
        candidate_vectors, relevance, sources_list,
        k=3, lambda_param=0.7, same_source_penalty=0.15
    )
    final_docs = [retrieved[i] for i in selected_indices]

    context = "\n\n".join([
        f"[{doc.metadata.get('source', 'guideline')}]\n{doc.page_content}"
        for doc in final_docs
    ])

    # build augmented prompt
    clinical_content = f"Clinical case:\n{case}\n\nQuestion: {question}" if case else f"Question: {question}"

    user_content = f"""{SYSTEM}

RELEVANT CLINICAL GUIDELINES:
{context}

{clinical_content}

Answer using the guidelines above. Cite the source document."""

    prompt = f"<|user|>\n{user_content}<|end|>\n<|assistant|>\n"

    output = pipe(prompt, max_new_tokens=500, do_sample=False)
    response = output[0]["generated_text"].split("<|assistant|>")[-1].strip()
    response = response.replace("<|end|>", "").strip()

    sources = [doc.metadata.get("source", "unknown") for doc in final_docs]
    return response, sources

# ── test on your worst questions ──────────────────────────────────────────────
TEST_QUESTIONS = [
    {
        "question": "What is the hydroxyurea monitoring protocol for adults with sickle cell disease?",
        "case": ""
    },
    {
        "question": "What is the diagnosis and immediate management in priority order?",
        "case": "A 4-year-old boy with HbSS presents with fever of 39.2°C and swollen tender hands and feet bilaterally."
    },
    {
        "question": "What are the indications for exchange transfusion in sickle cell disease?",
        "case": ""
    },
    {
        "question": "What monitoring intervals are recommended for adults stabilising on hydroxyurea?",
        "case": ""
    },
    {
        "question": "What are the newborn screening recommendations for sickle cell disease in Nigeria?",
        "case": ""
    },
]

if __name__ == "__main__":
    print("Loading model...")
    model, tokenizer = load_model()

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=500,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    print("Loading vector database...")
    vectordb, embeddings = load_vectordb()
    # vectordb.get() is Chroma's public API; pulling only "ids" keeps this
    # cheap since we don't need documents/embeddings/metadatas back just to
    # count them.
    chunk_count = len(vectordb.get(include=[])["ids"])
    print(f"Chunks loaded: {chunk_count}")

    print("Loading Cross-Encoder Reranker...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    cross_encoder = CrossEncoder("BAAI/bge-reranker-large", device=device)

    print("\nRunning RAG inference tests...\n")
    results = []

    for i, q in enumerate(TEST_QUESTIONS):
        print(f"[{i+1}/{len(TEST_QUESTIONS)}] {q['question'][:60]}...")
        try:
            response, sources = answer_with_rag(
                pipe, vectordb, embeddings, cross_encoder,
                q["question"],
                q["case"]
            )
        except Exception as e:
            # don't let one bad question kill the whole run / lose prior results
            print(f"  ERROR: {e}")
            response, sources = f"[ERROR: {e}]", []

        results.append({
            "question": q["question"],
            "case": q["case"],
            "response": response,
            "sources": sources
        })
        print(f"Sources: {sources}")
        print(f"Response: {response[:300]}\n")
        print("---")

        # write incrementally so a crash mid-run doesn't lose everything
        with open(OUTPUT_PATH, "w") as f:
            json.dump(results, f, indent=2)

    print(f"Done. Results saved to {OUTPUT_PATH}")

Loading model...


config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]


Running RAG inference tests...

[1/5] What is the hydroxyurea monitoring protocol for adults with ...
